In [1]:
# 1. Setup and Data Loading

import os, re, json, html
import pandas as pd
from graphviz import Digraph
import xml.etree.ElementTree as ET

# Config / file paths (same as original)
DATASET_PATH = "device_compromise_dataset.xlsx"
OUTPUT_DIR = "attack_tree_html_unified"
SVG_BASENAME = "comprehensive_attack_tree"
SVG_OUTPUT = os.path.join(OUTPUT_DIR, SVG_BASENAME + ".svg")
HTML_OUTPUT = os.path.join(OUTPUT_DIR, SVG_BASENAME + ".html")
PNG_OUTPUT = os.path.join(OUTPUT_DIR, SVG_BASENAME + ".png")

# Ensure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# MITRE tactic ordering (consistent ordering)
MITRE_TACTIC_ORDER = [
    "Reconnaissance", "Resource Development", "Initial Access", "Execution", "Persistence",
    "Privilege Escalation", "Defense Evasion", "Credential Access", "Discovery",
    "Lateral Movement", "Collection", "Command and Control", "Exfiltration", "Impact"
]

# Read the dataset
df = pd.read_excel(DATASET_PATH, index_col=False)
df = df.fillna("")  # replace NaN with empty strings

# The dataset must have a 'tactics' column
if "tactics" not in df.columns:
    raise KeyError("Expected 'tactics' column in the dataset (case-sensitive).")

df["tactics"] = df["tactics"].astype(str).str.strip()

# Determine ordered tactics present in the data
unique_tactics = [t for t in df["tactics"].unique() if t != ""]
ordered_tactics = [t for t in MITRE_TACTIC_ORDER if t in unique_tactics]
for t in unique_tactics:
    if t not in ordered_tactics:
        ordered_tactics.append(t)
print("Tactics (ordered):", ordered_tactics)


Tactics (ordered): ['Initial Access', 'Persistence', 'Privilege Escalation', 'Defense Evasion', 'Credential Access', 'Discovery', 'Lateral Movement', 'Collection']


In [2]:
# 2. Helper Functions

# Normalize column/key names for flexible matching
def _normalize_name(s: str) -> str:
    return re.sub(r'[^0-9a-z]', '', str(s).lower())

# Extract metadata by aggregating unique values in a DataFrame subset
def extract_node_metadata(df_subset: pd.DataFrame, node_type: str):
    cols_map = { _normalize_name(c): c for c in df_subset.columns.tolist() }
    if node_type == "cwe":
        keys = [
            'CVE ID', 'Description', 'cwe_Name', 'cwe_Abstraction', 'cwe_Structure', 'cwe_Status',
            'cwe_Description', 'cwe_Extended_Description', 'cwe_Related_Weaknesses', 'cwe_Weakness_Ordinality',
            'cwe_Language_Name', 'cwe_Technology_Class', 'cwe_Alternate_Terms', 'cwe_Alternate_Terms_With_Descriptions',
            'cwe_Background_Details', 'cwe_Modes_Of_Introduction', 'cwe_Likelihood_Of_Exploit',
            'cwe_Common_Consequences', 'cwe_Detection_Methods', 'cwe_Potential_Mitigations', 'cwe_Demonstrative_Examples',
            'cwe_Observed_Examples', 'attackVector', 'baseScore', 'baseSeverity', 'attackComplexity',
            'privilegesRequired', 'userInteraction', 'confidentialityImpact', 'availabilityImpact', 'exploitabilityScore'
        ]
    elif node_type == "capec":
        keys = [
            'CVE ID', 'capec_Name', 'capec_Mitre_Techniques', 'capec_Abstraction', 'capec_Status',
            'capec_Description', 'capec_Extended_Description', 'capec_Likelihood_Of_Attack', 'capec_Typical_Severity',
            'capec_Execution_Flow', 'capec_Prerequisites', 'capec_Skills_Required', 'capec_Resources_Required',
            'capec_Consequences', 'capec_Mitigations', 'capec_Example_Instances', 'capec_Related_Weaknesses',
            'capec_Taxonomy_Mappings'
        ]
    elif node_type == "tech":
        keys = [
            'CVE ID', 'Description', 'tech_description','technique_name', 'subtechnique_name', 'tactics', 'detection',
            'platforms', 'data sources', 'defenses bypassed', 'contributors', 'permissions required',
            'supports remote', 'system requirements', 'impact type', 'effective permissions',
            'attackVector', 'baseScore', 'baseSeverity', 'attackComplexity', 'privilegesRequired',
            'userInteraction', 'confidentialityImpact', 'availabilityImpact', 'exploitabilityScore'
        ]
    elif node_type == "subtech":
        keys = [
            'subtechnique_name', 'technique_name', 'tactics', 'detection', 'platforms', 'data sources'
        ]
    else:
        keys = df_subset.columns.tolist()

    meta = {}
    for k in keys:
        nk = _normalize_name(k)
        colname = cols_map.get(nk)
        if colname and colname in df_subset.columns:
            vals = df_subset[colname].astype(str).map(str.strip)
            vals = [v for v in vals if v != "" and v.lower() != "nan"]
            uniq = []
            for v in vals:
                if v not in uniq:
                    uniq.append(v)
            if len(uniq) == 1:
                meta[k] = uniq[0]
            elif len(uniq) > 1:
                meta[k] = " | ".join(uniq)
    return meta

# Format metadata dictionary into an HTML table string
def format_metadata_for_html(metadata_dict, max_rows=60):
    if not metadata_dict:
        return "<div class='meta-empty'>(No metadata available)</div>"
    rows = []
    # Preferred keys order
    preferred_order = ["cwe_Name", "capec_Name", "technique_name", "subtechnique_name", "Description", "CVE ID"]
    added = set()
    for k in preferred_order:
        if k in metadata_dict:
            rows.append((k, metadata_dict[k])); added.add(k)
    for k, v in metadata_dict.items():
        if k in added: continue
        if len(rows) >= max_rows: break
        rows.append((k, v))
    table_html = ["<table class='meta-table' style='width:100%;border-collapse:collapse;'>"]
    for k, v in rows:
        k_esc = html.escape(str(k))
        v_esc = html.escape(str(v))
        table_html.append(
            f"<tr><th style='text-align:left;padding:8px;border-bottom:1px solid #eee;"
            f"width:38%;vertical-align:top;background:#fafafa'>{k_esc}</th>"
            f"<td style='padding:8px;border-bottom:1px solid #eee;vertical-align:top'>{v_esc}</td></tr>"
        )
    table_html.append("</table>")
    return "\n".join(table_html)

# Gate-building functions (unchanged from original)
def build_gate_tree(dot, children, gate_label, gate_prefix, gate_counter, nodes_data, edges_data, gate_color="#FFCCCB"):
    if not children:
        return None
    if len(children) == 1:
        return children[0]
    def _build(sub_children):
        if len(sub_children) == 1:
            return sub_children[0]
        g_id = f"{gate_prefix}_{gate_counter[0]}"
        gate_counter[0] += 1
        dot.node(g_id, label=gate_label, shape="ellipse", fillcolor=gate_color, style="filled")
        nodes_data.append({"id": g_id, "label": gate_label, "type": "gate"})
        mid = len(sub_children) // 2
        left = _build(sub_children[:mid])
        right = _build(sub_children[mid:])
        if left:
            dot.edge(g_id, left); edges_data.append({"from": g_id, "to": left})
        if right:
            dot.edge(g_id, right); edges_data.append({"from": g_id, "to": right})
        return g_id
    return _build(children)

def binary_gate(dot, parent, children, gate_label, gate_color, gate_counter, nodes_data, edges_data, gate_prefix):
    if len(children) == 0:
        return
    if len(children) == 1:
        dot.edge(parent, children[0]); edges_data.append({"from": parent, "to": children[0]})
        return
    if len(children) == 2:
        g_id = f"{gate_prefix}_{gate_counter[0]}"
        gate_counter[0] += 1
        dot.node(g_id, label=gate_label, shape="ellipse", fillcolor=gate_color, style="filled")
        nodes_data.append({"id": g_id, "label": gate_label, "type": "gate"})
        dot.edge(parent, g_id); dot.edge(g_id, children[0]); dot.edge(g_id, children[1])
        edges_data.extend([
            {"from": parent, "to": g_id},
            {"from": g_id, "to": children[0]},
            {"from": g_id, "to": children[1]}
        ])
        return
    mid = len(children) // 2
    g_id = f"{gate_prefix}_{gate_counter[0]}"
    gate_counter[0] += 1
    dot.node(g_id, label=gate_label, shape="ellipse", fillcolor=gate_color, style="filled")
    nodes_data.append({"id": g_id, "label": gate_label, "type": "gate"})
    dot.edge(parent, g_id); edges_data.append({"from": parent, "to": g_id})
    binary_gate(dot, g_id, children[:mid], gate_label, gate_color, gate_counter, nodes_data, edges_data, gate_prefix)
    binary_gate(dot, g_id, children[mid:], gate_label, gate_color, gate_counter, nodes_data, edges_data, gate_prefix)


In [ ]:
# 3. Building the Attack Tree with Deduplication

def build_attack_tree():
    # Initialize graph
    dot = Digraph(name="comprehensive_attack_tree", format="svg")
    dot.attr('graph', rankdir='TB', splines='ortho', nodesep='0.6', ranksep='1.0')
    dot.attr('node', fontname='Helvetica', fontsize='28')
    dot.attr('edge', fontsize='8')

    nodes_data = []
    edges_data = []
    node_metadata_map = {}
    seen_paths = set()  # Set of (cwe, technique, subtech, capec) to skip duplicates

    # Root node
    root_id = "root"
    dot.node(root_id, label="Compromise Device", shape="box", fillcolor="#FFD700", style="filled")
    nodes_data.append({
        "id": root_id, "label": "Compromise Device", "type": "root",
        "metadata": {"Description": "Root node"}
    })

    # Tactic nodes
    tactic_ids = []
    for i, tactic in enumerate(ordered_tactics):
        tid = f"tactic_{i}"
        dot.node(tid, label=tactic, shape="box", fillcolor="#98FB98", style="filled")
        nodes_data.append({"id": tid, "label": tactic, "type": "tactic", "metadata": {"tactic": tactic}})
        tactic_ids.append(tid)

    # Connect root -> tactics (SAND gate)
    sand_counter = [0]
    binary_gate(dot, root_id, tactic_ids, "SAND", "#FFCCCB", sand_counter, nodes_data, edges_data, gate_prefix="sand")

    or_counter = [0]
    and_counter = [0]

    # For each tactic, attach CWE(s), then technique/subtech & CAPEC
    for ti, tactic in enumerate(ordered_tactics):
        tid = f"tactic_{ti}"
        tactic_df = df[df["tactics"] == tactic]
        # Unique CWEs in this tactic
        cwe_names = [x for x in tactic_df.get("cwe_Name", pd.Series([], dtype=object))
                     .unique().tolist() if x != ""]
        cwe_ids = []
        for ci, cwe in enumerate(cwe_names):
            cwe_id = f"cwe_{ti}_{ci}"
            cwe_subset = tactic_df[tactic_df["cwe_Name"] == cwe]
            meta = extract_node_metadata(cwe_subset, "cwe")
            dot.node(cwe_id, label=cwe, shape="note", fillcolor="#ADD8E6", style="filled")
            nodes_data.append({"id": cwe_id, "label": cwe, "type": "cwe", "metadata": meta})
            node_metadata_map[cwe_id] = format_metadata_for_html(meta)
            cwe_ids.append(cwe_id)
        # Connect tactic -> CWE(s) via OR gate
        if cwe_ids:
            binary_gate(dot, tid, cwe_ids, "OR", "#FFCCCB", or_counter, nodes_data, edges_data, gate_prefix="or")
        
        # For each CWE, process its techniques and subtechniques
        for ci, cwe in enumerate(cwe_names):
            cwe_id = f"cwe_{ti}_{ci}"
            cwe_subset = tactic_df[tactic_df["cwe_Name"] == cwe]
            techniques = [x for x in cwe_subset.get("technique_name", pd.Series([], dtype=object))
                          .unique().tolist() if x != ""]
            for t_idx, tech in enumerate(techniques):
                tech_subset = cwe_subset[cwe_subset["technique_name"] == tech]
                subtechs = [s for s in tech_subset.get("subtechnique_name", pd.Series([], dtype=object))
                            .unique().tolist() if s != ""]

                if subtechs:
                    for s_i, subtech in enumerate(subtechs):
                        sub_rows = tech_subset[tech_subset["subtechnique_name"] == subtech]
                        # Find all CAPECs for this subtech
                        capec_names = [c for c in sub_rows.get("capec_Name", pd.Series([], dtype=object))
                                       .unique().tolist() if c != ""]
                        # If no CAPECs, plan to add placeholder
                        if not capec_names:
                            capec_candidates = ["CAPEC ?"]
                        else:
                            capec_candidates = capec_names

                        # Filter out already-seen paths
                        new_capecs = []
                        for cap in capec_candidates:
                            path = (cwe, tech, subtech, cap)
                            if path in seen_paths:
                                continue
                            seen_paths.add(path)
                            new_capecs.append(cap)

                        # Skip subtech entirely if no new CAPECs
                        if not new_capecs:
                            continue

                        # Create subtech node (now that it has new CAPEC children)
                        sub_id = f"subtech_{ti}_{ci}_{t_idx}_{s_i}"
                        sub_meta = extract_node_metadata(sub_rows, "subtech")
                        dot.node(sub_id, label=subtech, shape="box", fillcolor="#FFFACD", style="filled")
                        nodes_data.append({"id": sub_id, "label": subtech, "type": "subtech", "metadata": sub_meta})
                        node_metadata_map[sub_id] = format_metadata_for_html(sub_meta)

                        # Create CAPEC nodes for new CAPECs
                        capec_nodes = []
                        for cap_i, cap in enumerate(new_capecs):
                            cap_id = f"capec_{ti}_{ci}_{t_idx}_{s_i}_{cap_i}"
                            cap_rows = sub_rows[sub_rows["capec_Name"] == cap]
                            cap_meta = extract_node_metadata(cap_rows, "capec")
                            dot.node(cap_id, label=cap, shape="box", fillcolor="#FFB6C1", style="filled")
                            nodes_data.append({"id": cap_id, "label": cap, "type": "capec", "metadata": cap_meta})
                            node_metadata_map[cap_id] = format_metadata_for_html(cap_meta)
                            capec_nodes.append(cap_id)

                        # If for some reason capec_nodes is empty (shouldn't be, as we ensured new_capecs),
                        # we could add a placeholder, but this case should not occur here.

                        # Build an OR gate tree among the CAPEC nodes (to allow multiple)
                        cap_rep_node = build_gate_tree(dot, capec_nodes, "OR", "orcap", or_counter, nodes_data, edges_data)

                        # Connect CWE -> (subtech & CAPEC-gate) via an AND gate
                        and_id = f"and_{ti}_{ci}_{t_idx}_{s_i}_{and_counter[0]}"
                        and_counter[0] += 1
                        dot.node(and_id, label="AND", shape="ellipse", fillcolor="#FFCCCB", style="filled")
                        nodes_data.append({"id": and_id, "label": "AND", "type": "gate"})
                        dot.edge(cwe_id, and_id); edges_data.append({"from": cwe_id, "to": and_id})
                        dot.edge(and_id, sub_id); edges_data.append({"from": and_id, "to": sub_id})
                        if cap_rep_node:
                            dot.edge(and_id, cap_rep_node); edges_data.append({"from": and_id, "to": cap_rep_node})
                else:
                    # No subtechniques: treat tech as direct child
                    tech_id = f"tech_{ti}_{ci}_{t_idx}"
                    # Find all CAPECs for this tech
                    capec_names = [c for c in tech_subset.get("capec_Name", pd.Series([], dtype=object))
                                   .unique().tolist() if c != ""]
                    if not capec_names:
                        capec_candidates = ["CAPEC ?"]
                    else:
                        capec_candidates = capec_names

                    new_capecs = []
                    for cap in capec_candidates:
                        path = (cwe, tech, "", cap)
                        if path in seen_paths:
                            continue
                        seen_paths.add(path)
                        new_capecs.append(cap)

                    # If no new CAPECs, skip this tech
                    if not new_capecs:
                        continue

                    # Create tech node (it has new CAPEC children)
                    tech_meta = extract_node_metadata(tech_subset, "tech")
                    dot.node(tech_id, label=tech, shape="box", fillcolor="#90EE90", style="filled")
                    nodes_data.append({"id": tech_id, "label": tech, "type": "tech", "metadata": tech_meta})
                    node_metadata_map[tech_id] = format_metadata_for_html(tech_meta)

                    capec_nodes = []
                    for cap_i, cap in enumerate(new_capecs):
                        cap_id = f"capec_{ti}_{ci}_{t_idx}_tcap_{cap_i}"
                        cap_rows = tech_subset[tech_subset["capec_Name"] == cap]
                        cap_meta = extract_node_metadata(cap_rows, "capec")
                        dot.node(cap_id, label=cap, shape="box", fillcolor="#FFB6C1", style="filled")
                        nodes_data.append({"id": cap_id, "label": cap, "type": "capec", "metadata": cap_meta})
                        node_metadata_map[cap_id] = format_metadata_for_html(cap_meta)
                        capec_nodes.append(cap_id)

                    # OR tree among CAPECs
                    cap_rep_node = build_gate_tree(dot, capec_nodes, "OR", "orcap", or_counter, nodes_data, edges_data)

                    # Connect CWE -> (tech & CAPEC-gate) via AND
                    and_id = f"and_{ti}_{ci}_{t_idx}_tech_{and_counter[0]}"
                    and_counter[0] += 1
                    dot.node(and_id, label="AND", shape="ellipse", fillcolor="#FFCCCB", style="filled")
                    nodes_data.append({"id": and_id, "label": "AND", "type": "gate"})
                    dot.edge(cwe_id, and_id); edges_data.append({"from": cwe_id, "to": and_id})
                    dot.edge(and_id, tech_id); edges_data.append({"from": and_id, "to": tech_id})
                    if cap_rep_node:
                        dot.edge(and_id, cap_rep_node); edges_data.append({"from": and_id, "to": cap_rep_node})

    # After building nodes/edges, propagate reachable nodes metadata to tactics
    adj = {}
    for e in edges_data:
        u = e.get("from"); v = e.get("to")
        if u is None or v is None: 
            continue
        adj.setdefault(u, []).append(v)

    id_to_type  = { nd["id"]: nd.get("type", "") for nd in nodes_data }
    id_to_label = { nd["id"]: nd.get("label", nd["id"]) for nd in nodes_data }

    def reachable_nodes(start_id):
        seen = set([start_id])
        queue = [start_id]
        result = []
        while queue:
            cur = queue.pop(0)
            for nb in adj.get(cur, []):
                if nb not in seen:
                    seen.add(nb)
                    result.append(nb)
                    queue.append(nb)
        return result

    for i, tactic in enumerate(ordered_tactics):
        tid = f"tactic_{i}"
        reached = reachable_nodes(tid)
        cwe_labels = []
        tech_labels = []
        capec_labels = []
        for nid in reached:
            ntype = id_to_type.get(nid, "")
            nlabel = id_to_label.get(nid, nid)
            if ntype == "cwe":
                if nlabel not in cwe_labels: cwe_labels.append(nlabel)
            elif ntype in ("tech", "subtech"):
                if nlabel not in tech_labels: tech_labels.append(nlabel)
            elif ntype == "capec":
                if nlabel not in capec_labels: capec_labels.append(nlabel)
        tactic_meta = {
            "tactic": tactic,
            "cwes": ", ".join(cwe_labels) if cwe_labels else "",
            "techniques": ", ".join(tech_labels) if tech_labels else "",
            "capecs": ", ".join(capec_labels) if capec_labels else ""
        }
        # Update nodes_data and node_metadata_map for tactic
        for nd in nodes_data:
            if nd["id"] == tid:
                nd_meta = nd.get("metadata", {}).copy()
                nd_meta.update(tactic_meta)
                nd["metadata"] = nd_meta
                break
        node_metadata_map[tid] = format_metadata_for_html(tactic_meta)

    return dot, nodes_data, edges_data, node_metadata_map


In [4]:
# 4. Graph Rendering and Output

# Main execution: build graph and save files
dot, nodes_data, edges_data, node_metadata_map = build_attack_tree()
basepath = os.path.join(OUTPUT_DIR, SVG_BASENAME)

# Render to SVG via Graphviz (requires 'dot' executable)
dot.render(basepath, format="svg", cleanup=True)
svg_path = basepath + ".svg"

# Inject node IDs into SVG and extract coordinates for later use
def inject_tooltips_and_extract_positions(svg_path, node_metadata_map):
    ns = {'svg': 'http://www.w3.org/2000/svg'}
    ET.register_namespace("", ns['svg'])
    tree = ET.parse(svg_path)
    root = tree.getroot()
    node_positions = {}
    for g in root.findall(".//svg:g", ns):
        title = g.find("svg:title", ns)
        if title is None: continue
        original_title_text = title.text or ""
        node_id = original_title_text
        # Use the visible label text as tooltip
        tt = g.find(".//svg:text", ns)
        label_text = tt.text if tt is not None and tt.text is not None else node_id
        title.text = label_text
        if node_id:
            g.set("data-node-id", node_id)
        transform = g.get("transform", "")
        m = re.search(r"translate\(\s*([0-9\.\-]+)[,\s]+([0-9\.\-]+)\s*\)", transform)
        if m:
            try:
                x, y = float(m.group(1)), float(m.group(2))
                node_positions[node_id] = {"x": x, "y": y}
            except:
                pass
    tree.write(svg_path, encoding="utf-8", xml_declaration=True)
    return node_positions

node_positions = inject_tooltips_and_extract_positions(svg_path, node_metadata_map)

# Save HTML wrapper with embedded SVG and metadata panel
def save_html_with_svg(svg_path, html_path, node_positions, nodes_data, node_metadata_map):
    with open(svg_path, "r", encoding="utf-8") as f:
        svg_content = f.read()
    meta_json = json.dumps(node_metadata_map)
    positions_json = json.dumps(node_positions)
    nodes_data_json = json.dumps(nodes_data)

    script_blob = f"""
<script>
window.ATTACK_TREE_META = {{
  "node_positions": {positions_json},
  "short_metadata": {meta_json},
  "nodes_data": {nodes_data_json}
}};
document.addEventListener('DOMContentLoaded', function() {{
  let panelRoot = document.getElementById('panel-root');
  if (!panelRoot) {{
    panelRoot = document.createElement('div');
    panelRoot.id = 'panel-root';
    document.body.appendChild(panelRoot);
  }}
  let panel = document.createElement('div');
  panel.id = 'metadata-panel';
  panel.style.boxSizing = 'border-box';
  panel.style.height = '100%';
  panel.style.width = '100%';
  panel.style.background = '#fff';
  panel.style.overflow = 'auto';
  panel.style.padding = '14px';
  panel.style.fontFamily = 'Arial, Helvetica, sans-serif';
  panel.style.fontSize = '15px';
  panel.style.display = 'none';
  panelRoot.appendChild(panel);

  let header = document.createElement('div');
  header.style.display = 'flex';
  header.style.alignItems = 'center';
  header.style.justifyContent = 'space-between';
  header.style.marginBottom = '10px';
  let title = document.createElement('div');
  title.innerText = 'Node metadata';
  title.style.fontWeight = '700';
  title.style.fontSize = '18px';
  header.appendChild(title);
  let closeBtn = document.createElement('button');
  closeBtn.innerHTML = '&times;';
  closeBtn.title = 'Close';
  closeBtn.style.border = 'none';
  closeBtn.style.background = 'transparent';
  closeBtn.style.fontSize = '24px';
  closeBtn.style.cursor = 'pointer';
  closeBtn.style.lineHeight = '1';
  closeBtn.style.padding = '0 6px';
  closeBtn.addEventListener('click', function() {{ panel.style.display = 'none'; }});
  header.appendChild(closeBtn);
  panel.appendChild(header);

  let content = document.createElement('div');
  content.id = 'metadata-panel-content';
  panel.appendChild(content);

  let foot = document.createElement('div');
  foot.style.marginTop = '12px';
  foot.style.fontSize = '13px';
  foot.style.color = '#666';
  foot.innerText = 'Click a node in the graph to view metadata here. Press × to close.';
  panel.appendChild(foot);

  let svgWrap = document.querySelector('.svg-wrap');
  if (!svgWrap) return;
  let nodeGroups = svgWrap.querySelectorAll('[data-node-id]');
  nodeGroups.forEach(function(g) {{
    g.style.cursor = 'pointer';
    g.addEventListener('click', function(ev) {{
      ev.stopPropagation();
      let nodeId = g.getAttribute('data-node-id');
      let html = window.ATTACK_TREE_META.short_metadata[nodeId] || "";
      if (!html || html === "") {{
        let found = (window.ATTACK_TREE_META.nodes_data || []).find(x => x.id === nodeId);
        if (found && found.label) {{
          html = '<div style="font-weight:600;margin-bottom:8px">' + found.label + '</div>';
        }} else {{
          html = '<div class="meta-empty">(No metadata)</div>';
        }}
      }}
      let subtitle = '<div style="color:#555;font-size:13px;margin-bottom:10px">ID: ' + nodeId + '</div>';
      content.innerHTML = subtitle + html;
      panel.style.display = 'block';
      panel.scrollTop = 0;
    }});
  }});

  document.addEventListener('keydown', function(e) {{
    if (e.key === 'Escape') panel.style.display = 'none';
  }});
}});
</script>
"""
    html_doc = f"""<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8" />
  <title>Comprehensive Attack Tree</title>
  <style>
    html, body {{ height: 100%; margin: 0; padding: 0; }}
    body {{ font-family: Arial, Helvetica, sans-serif; background: #fafafa; font-size:16px; }}
    .app {{ display:flex; height: calc(100vh - 20px); gap: 12px; padding: 10px; box-sizing: border-box; }}
    .main {{ flex: 1 1 auto; display:flex; flex-direction: column; min-width: 200px; }}
    .topbar {{ padding-bottom: 8px; }}
    h1 {{ font-size: 24px; margin: 0 0 6px 0; }}
    .legend {{ margin-top: 6px; font-size: 15px; }}
    .legend span {{ display: inline-block; margin-right: 12px; padding: 4px 8px; border-radius:4px; font-size:14px; }}
    p.note {{ font-size: 15px; color: #333; margin-top:8px; }}
    .svg-wrap {{ flex: 1 1 auto; border: 1px solid #ddd; padding: 6px; background: #fff; overflow: auto; min-height: 200px; }}
    .svg-wrap svg {{ height: 100%; width: auto; display: block; }}
    #panel-root {{ flex: 0 0 420px; max-width: 42vw; min-width: 320px; display:flex; flex-direction:column; box-sizing: border-box; }}
    @media (max-width: 1000px) {{
      #panel-root {{ min-width: 260px; max-width: 40vw; }}
      .legend span {{ font-size:13px; }}
    }}
    .meta-table th {{ background:#fafafa; font-size:14px; }}
    .meta-table td {{ font-size:14px; }}
    .meta-empty {{ color:#666; font-style:italic; }}
  </style>
</head>
<body>
  <div class="app">
    <div class="main">
      <div class="topbar">
        <h1>Comprehensive Attack Tree</h1>
        <div class="legend">
          <span style="background:#FFD700">Root</span>
          <span style="background:#98FB98">Tactic</span>
          <span style="background:#ADD8E6">CWE</span>
          <span style="background:#90EE90">Technique</span>
          <span style="background:#FFFACD">Subtechnique</span>
          <span style="background:#FFB6C1">CAPEC</span>
          <span style="background:#FFCCCB">Gates</span>
        </div>
      </div>

      <div class="svg-wrap">
        {svg_content}
      </div>

      <p class="note">Click a node to view metadata in the right-side panel. The attack tree canvas fills the remaining screen area for easier viewing.</p>
    </div>

    <!-- placeholder for metadata panel -->
    <div id="panel-root" aria-hidden="false"></div>
  </div>

  {script_blob}
</body>
</html>
"""
    with open(html_path, "w", encoding="utf-8") as f:
        f.write(html_doc)

save_html_with_svg(svg_path, HTML_OUTPUT, node_positions, nodes_data, node_metadata_map)

# Optionally render PNG (if Graphviz supports it)
try:
    dot.render(basepath, format="png", cleanup=True)
except Exception as e:
    print("PNG rendering skipped (optional).", e)

print("Outputs generated:")
print(" - SVG:", svg_path)
print(" - HTML:", HTML_OUTPUT)
print(" - PNG (optional):", basepath + ".png")


Outputs generated:
 - SVG: attack_tree_html_unified/comprehensive_attack_tree.svg
 - HTML: attack_tree_html_unified/comprehensive_attack_tree.html
 - PNG (optional): attack_tree_html_unified/comprehensive_attack_tree.png
